# Correlation Power Analysis (Brier et al. 2004)

In [1]:
%load_ext autoreload
%autoreload 2

import os
import random

import lascar
import numpy as np
import plotly.graph_objects as pgo
from cwtoolbox import CaptureDevice

In [2]:
device = CaptureDevice.create("CWLITEXMEGA")
device.compile(file=os.path.abspath("../lecture_3/sbox_lookup.c"))
device.flash()

c:\work\securecoding_ws2526\.venv\Lib\site-packages\chipwhisperer\capture\trace\TraceWhisperer.py:31: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources # type: ignore


XMEGA Programming flash...
XMEGA Reading flash...
Verified flash OK, 2553 bytes


In [3]:
data = device.capture(
    number_of_traces=500,
    input=lambda _: random.randbytes(16)
)

100%|██████████| 500/500 [00:10<00:00, 49.70it/s]


In [4]:
# Create a dict with keys 0,...,8 and values containing
# lists of traces where the hamming weight of the first byte equals the key
grouped_data_raw = {
    hw: [d for d in data if lascar.hamming_weight(d["input"][0]) == hw]
    for hw in range(9)
}

In [5]:
# Calculate the average trace for each group
grouped_data = {
    hw: np.mean(np.array(traces)["trace"], axis=0)
    for hw, traces in grouped_data_raw.items()
}

In [6]:
# Plot the mean traces
fig = pgo.Figure()
for hw, trace in grouped_data.items():
    fig.add_trace(pgo.Scatter(y=trace, name=f"HW: {hw}"))
fig.show()


In [7]:
# Plot hw vs trace point
# The trace point is selected where "the traces look different".
fig = pgo.Figure()
for hw, trace in grouped_data.items():
    fig.add_trace(pgo.Scatter(x=[hw], y=[trace[61]]))
fig.show()
# => We see that the hamming weight of the input is proportional
# to the current consumption in the moment where the input is processed.

## 2. Pearson correlation coefficient

An interesting statistical formula to face this problem is given by the *Pearson correlation coefficient*. For two random variables $X, Y$ it is defined as

$$\rho_{X,Y} := \frac{\mathrm{Cov}(X, Y)}{\sqrt{\mathrm{Var}(X)} \sqrt{\mathrm{Var}(Y)}} \ \in [-1, 1]\,.$$

For two samples of finite length $x = {x_1, ..., x_n}$, $y = {y_1, ..., y_n}$ it can be defined as 

$$r_{x,y} := \frac{\sum_{i=1}^n (x_i - \bar x)(y_i - \bar y)}{\sqrt{\sum_{i=1}^n (x_i - \bar x)^2}\sqrt{\sum_{i=1}^n (y_i - \bar y)^2}} \ \in [-1, 1]\,,$$

where $\bar x := \frac{1}{n} \sum_{i=1}^n x_i$ is the mean of a sample $x$.

In [8]:
def pearson(xs, ys):
    xmean = np.mean(xs)
    ymean = np.mean(ys)
    return sum((xs - xmean) * (ys - ymean)) / np.sqrt(
        sum((xs - xmean) ** 2) * sum((ys - ymean) ** 2)
    )


fig = pgo.Figure()
size = 50
data1 = 5 * np.array(range(size)) + np.random.uniform(-size / 4, size / 4, size=size)
fig.add_trace(pgo.Scatter(y=data1, name=f"pearson:{pearson(range(size), data1)}"))
data2 = np.array(range(size)) + np.random.uniform(-size, size, size=size) + 10
fig.add_trace(pgo.Scatter(y=data2, name=f"pearson:{pearson(range(size), data2)}"))
data3 = (
    -3 * np.array(range(size)) + np.random.uniform(-size / 4, size / 4, size=size) + 200
)
fig.add_trace(pgo.Scatter(y=data3, name=f"pearson:{pearson(range(size), data3)}"))
data4 = 100 * np.sin(np.array(range(size)) / size * np.pi + np.pi) + np.random.uniform(
    -size / 10, size / 10, size=size
)
fig.add_trace(pgo.Scatter(y=data4, name=f"pearson:{pearson(range(size), data4)}"))
fig.show()

## 3. Attack!

In [9]:
import lascar.tools


def aes_sbox_cpa(traces, key_byte_index=0, trace_point=145):
    # All trace values at the given trace point
    values_at_trace_point = traces["trace"][:, trace_point]
    # Iterate over all possible keys
    pearsons = []
    for guess in range(256):
        # Hamming weights of hypothesis
        hamming_weights = [
            lascar.hamming(lascar.tools.aes.sbox[inp[key_byte_index] ^ guess])
            for inp in traces["input"]
        ]
        # Calculate pearson of these two
        p = pearson(values_at_trace_point, hamming_weights)
        pearsons.append(abs(p))
    print(np.argmax(pearsons))


aes_sbox_cpa(traces=data)

1


In [10]:
# Get rid of the trace point as input
def aes_sbox_cpa2(traces, key_byte_index=0):
    # Iterate over all possible keys
    pearsons = []
    for guess in range(256):
        # Iterate over all trace points
        pearsons_per_point = []
        for trace_point in range(traces["trace"].shape[1]):
            # All trace values at the given trace point
            values_at_trace_point = traces["trace"][:, trace_point]
            # Hamming weights of hypothesis
            hamming_weights = [
                lascar.hamming(lascar.tools.aes.sbox[inp[key_byte_index] ^ guess])
                for inp in traces["input"]
            ]
            # Calculate pearson of these two
            p = pearson(values_at_trace_point, hamming_weights)
            pearsons_per_point.append(abs(p))
        # Store the maximum pearson coefficient of all trace points for the current guess
        print(f"Max pearson for guess {guess}: {max(pearsons_per_point):.02f}")
        pearsons.append(max(pearsons_per_point))
    print(np.argmax(pearsons))

aes_sbox_cpa2(traces=data[:20], key_byte_index=7)

Max pearson for guess 0: 0.60
Max pearson for guess 1: 0.67
Max pearson for guess 2: 0.80
Max pearson for guess 3: 0.65
Max pearson for guess 4: 0.74
Max pearson for guess 5: 0.67
Max pearson for guess 6: 0.63
Max pearson for guess 7: 0.67
Max pearson for guess 8: 0.91
Max pearson for guess 9: 0.62
Max pearson for guess 10: 0.76
Max pearson for guess 11: 0.68
Max pearson for guess 12: 0.66
Max pearson for guess 13: 0.71
Max pearson for guess 14: 0.68
Max pearson for guess 15: 0.75
Max pearson for guess 16: 0.76
Max pearson for guess 17: 0.61
Max pearson for guess 18: 0.69
Max pearson for guess 19: 0.68
Max pearson for guess 20: 0.68
Max pearson for guess 21: 0.77
Max pearson for guess 22: 0.64
Max pearson for guess 23: 0.72
Max pearson for guess 24: 0.72
Max pearson for guess 25: 0.77
Max pearson for guess 26: 0.66
Max pearson for guess 27: 0.63
Max pearson for guess 28: 0.61
Max pearson for guess 29: 0.61
Max pearson for guess 30: 0.60
Max pearson for guess 31: 0.64
Max pearson for gu

<div style="border: 3px solid plum; border-radius: 5px; padding: 5px; width: calc(100% - 20px);">
<div class="h2" style="font-variant: all-small-caps;">Exercise 5</div>

Perform a CPA attack using Lascar's `CpaEngine`.

</div>